# Single-cell RNA-seq workshop (Scanpy + PBMC3k)

This notebook is a workshop starter built from the Scanpy PBMC tutorial workflow (Scanpy 1.10 docs).

## Learning goals
- Load PBMC data into an AnnData object
- Run a standard Scanpy preprocessing workflow
- Build UMAP and cluster cells
- Identify marker genes and annotate clusters

> Workshop note: this notebook is intentionally written as an outline with runnable starter code. Expand each section with discussion prompts and interpretation during the workshop.


## 0) Setup
Install (if needed): `pip install scanpy anndata matplotlib seaborn`

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

sc.settings.verbosity = 2
sc.set_figure_params(dpi=100, facecolor='white')
print(sc.__version__)

## 1) Load PBMC data
Using the PBMC dataset referenced in the Scanpy tutorials.

In [ ]:
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()
adata

## 2) Quality control
Explore counts/genes per cell and mitochondrial content.

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter=0.4, multi_panel=True)
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
# Common PBMC tutorial-style filters (adjust during workshop)
adata = adata[adata.obs.n_genes_by_counts < 2500, :].copy()
adata = adata[adata.obs.pct_counts_mt < 5, :].copy()
adata

## 3) Normalization, feature selection, scaling

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.raw = adata

sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)

adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata, max_value=10)

## 4) Dimensionality reduction and clustering

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True)

sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

sc.pl.umap(adata, color=['leiden'])

## 5) Marker genes and interpretation

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

marker_table = sc.get.rank_genes_groups_df(adata, group=None)
marker_table.head()

## 6) Suggested workshop exercises
1. Change filtering thresholds and compare cluster structure.
2. Test multiple `resolution` values for Leiden clustering.
3. Use canonical PBMC marker genes to annotate clusters manually.
4. Save figures and a processed AnnData object for downstream differential analysis.

In [ ]:
# Optional save step
adata.write('pbmc3k_workshop_processed.h5ad')